In [ ]:

from pathlib import Path
import shutil #copyfile() needed

# Job parameters
dbutils.widgets.text("catalog", "")
dbutils.widgets.text("bronze_schema", "")
dbutils.widgets.text("raw_volume", "raw_files")
dbutils.widgets.text("source_data_path", "")

catalog = dbutils.widgets.get("catalog").strip()
schema = dbutils.widgets.get("bronze_schema").strip()
volume = dbutils.widgets.get("raw_volume").strip()
source_path = dbutils.widgets.get("source_data_path").strip()

if not all([catalog, schema, volume, source_path]):
    raise ValueError("Missing required Bootstrap parameters")

# Input files synchronized by Databricks Bundle
source_dir = Path(source_path)

# Destination Volume for the selected environment
target_dir = Path(f"/Volumes/{catalog}/{schema}/{volume}")

required_files = [
    "europe_cereal_yield_climate_1990_2022.csv",
    "environment_clean.csv",
]

# Validate all source files before copying
for filename in required_files:
    source = source_dir / filename
    if not source.is_file() or source.stat().st_size == 0:
        raise FileNotFoundError(
            f"Missing or empty source file: {source}"
        )

# Validate the target Volume
if not target_dir.is_dir():
    raise FileNotFoundError(f"Unity Catalog Volume does not exist: {target_dir}")

for filename in required_files:
    source = source_dir / filename
    target = target_dir / filename

    if target.is_file():
        if target.stat().st_size != source.stat().st_size:
            raise RuntimeError(
                f"Existing target differs in size: {target}. "
                "Manual inspection is required."
            )
        print(f"SKIP: {filename} already exists")
        continue

    shutil.copyfile(source, target)

    if target.stat().st_size != source.stat().st_size:
        raise RuntimeError(f"Copy verification failed: {filename}")

    print(f"COPIED: {filename}")

print(f"Bootstrap completed successfully for {catalog}.{schema}")
